<a href="https://colab.research.google.com/github/AngelTroncoso/Alergias/blob/main/Open_Target.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Descarga de Datos de Open Target

In [9]:
# Instalar lftp, una herramienta robusta para mirroring de FTP/HTTP.
# Es recomendada por Open Targets para descargas de datasets.

In [12]:
!apt-get update
!apt-get install -y lftp

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,978 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,960 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,310 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,622 kB]
Get:14 http://security.

Ahora, copiaremos los datos de evidencia de ClinVar (EVA) de la versión `25.03` de Open Targets desde el servidor FTP de Open Targets a tu entorno de Colab usando `wget`. Esto puede tardar unos minutos, dependiendo del tamaño del conjunto de datos.

In [14]:
# Define la ruta del servidor FTP al conjunto de datos de ejemplo
ftp_path = "https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.03/output/evidence/sourceId=eva"
# Define la ruta local donde se copiarán los datos
local_path = "/content/opentargets_evidence_eva"

print(f"Copiando datos de {ftp_path} a {local_path} usando lftp...")

# Crea el directorio local si no existe
!mkdir -p {local_path}

# Usa lftp para copiar el directorio recursivamente.
# 'mirror' sincroniza un directorio remoto con uno local.
# '--verbose' muestra más detalles del progreso.
# '--exclude-glob' permite excluir patrones de archivos no deseados (como los archivos HTML de listado de directorios).
!lftp -c "open {ftp_path}; mirror --verbose --exclude-glob *.html --exclude-glob *.txt --no-symlinks --dereference --only-newer --allow-chown --allow-suid --scan-all --continue . {local_path}"

print("Copia de datos completada.")

Copiando datos de https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.03/output/evidence/sourceId=eva a /content/opentargets_evidence_eva usando lftp...
cd: received redirection to `https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.03/output/evidence/sourceId=eva/'
Transferring file `part-00200-3b3d3d5d-6701-4d25-96ef-b12dc2ca1d97.c000.snappy.parquet'
Transferring file `part-00201-3b3d3d5d-6701-4d25-96ef-b12dc2ca1d97.c000.snappy.parquet'
Transferring file `part-00202-3b3d3d5d-6701-4d25-96ef-b12dc2ca1d97.c000.snappy.parquet'
Transferring file `part-00203-3b3d3d5d-6701-4d25-96ef-b12dc2ca1d97.c000.snappy.parquet'
Transferring file `part-00204-3b3d3d5d-6701-4d25-96ef-b12dc2ca1d97.c000.snappy.parquet'
Transferring file `part-00205-3b3d3d5d-6701-4d25-96ef-b12dc2ca1d97.c000.snappy.parquet'
Transferring file `part-00206-3b3d3d5d-6701-4d25-96ef-b12dc2ca1d97.c000.snappy.parquet'
Transferring file `part-00207-3b3d3d5d-6701-4d25-96ef-b12dc2ca1d97.c000.snappy.parquet'
Transferring

Una vez copiados los archivos Parquet localmente, podemos cargarlos en un DataFrame de pandas.

In [16]:
import pandas as pd
import os
import glob # Import glob to find parquet files

# Asegúrate de que la ruta local existe antes de intentar leer
if os.path.exists(local_path):
    print(f"Leyendo archivos Parquet desde {local_path}...")

    # Obtener una lista de todos los archivos .parquet en el directorio
    parquet_files = glob.glob(os.path.join(local_path, "*.parquet"))

    if not parquet_files:
        print(f"Error: No se encontraron archivos Parquet en el directorio {local_path}.")
    else:
        # Leer el conjunto de datos Parquet en un DataFrame de pandas
        # pandas.read_parquet puede manejar directorios que contienen múltiples archivos parquet,
        # pero para ser robustos ante archivos no Parquet, pasamos solo los archivos .parquet explícitamente.
        df = pd.read_parquet(parquet_files)

        print("¡Datos cargados exitosamente!")
        print("Mostrando las primeras 5 filas:")
        display(df.head())
        print("\nInformación del DataFrame:")
        df.info()
else:
    print(f"Error: El directorio {local_path} no se encontró. La copia de datos podría haber fallado.")

Leyendo archivos Parquet desde /content/opentargets_evidence_eva...
¡Datos cargados exitosamente!
Mostrando las primeras 5 filas:


,datasourceId,targetId,alleleOrigins,allelicRequirements,ancestry,ancestryId,beta,betaConfidenceIntervalLower,betaConfidenceIntervalUpper,biologicalModelAllelicComposition,...,variantRsId,pmcIds,publicationYear,studyLocusId,textMiningSentences,diseaseId,id,score,variantEffect,directionOnTrait
0,eva,ENSG00000000971,[germline],None,None,None,NaN,NaN,NaN,None,...,rs35695425,None,NaN,None,None,MONDO_0012540,1babef1341e54add71e3d20bfa1102b7293dcfb9,0.02,None,None
1,eva,ENSG00000000971,[germline],None,None,None,NaN,NaN,NaN,None,...,rs1137971,None,NaN,None,None,MONDO_0012540,5fe2e371a4ab298c32da902589476ce59d2e96c4,0.02,None,None
2,eva,ENSG00000000971,None,None,None,None,NaN,NaN,NaN,None,...,rs779166622,None,NaN,None,None,MONDO_0012540,58bb5cc979af8713674f98b0c12e72b0ce005f70,0.32,None,None
3,eva,ENSG00000000971,[germline],None,None,None,NaN,NaN,NaN,None,...,rs886045741,None,NaN,None,None,MONDO_0012540,e585f70287fdd57c310ec27bd19cc1ca94d3d383,0.32,None,None
4,eva,ENSG00000000971,[germline],None,None,None,NaN,NaN,NaN,None,...,rs15809,None,NaN,None,None,MONDO_0012540,150f51aeb449dd6fd40f9d714fcce88edc474e48,0.05,None,None



Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3138383 entries, 0 to 3138382
Data columns (total 88 columns):
 #   Column                                 Dtype  
---  ------                                 -----  
 0   datasourceId                           object 
 1   targetId                               object 
 2   alleleOrigins                          object 
 3   allelicRequirements                    object 
 4   ancestry                               object 
 5   ancestryId                             object 
 6   beta                                   float64
 7   betaConfidenceIntervalLower            float64
 8   betaConfidenceIntervalUpper            float64
 9   biologicalModelAllelicComposition      object 
 10  biologicalModelGeneticBackground       object 
 11  biologicalModelId                      object 
 12  biomarkerName                          object 
 13  biomarkers                             object 
 14  biosamplesFromSource  